# A5 — per-subject membership inference

Threat: an adversary with black-box access to a deployed BCI decoder asks, for each candidate subject, **was their data used to train this model?**

Standard Shokri-style shadow-model attack adapted to per-subject membership: 20 shadow EEGNets on random 50% subject splits, attack classifier on (mean-loss, mean-confidence) features, evaluation on a held-out target model. Reports AUC and the (TPR − FPR) attack advantage.

Send back `results/08_a5_membership_inference.json` and `runs/<run_id>/meta.json`.

**Runtime → L4 GPU.** Expected wall ~30 min.

**Don't `Save a copy in GitHub` from Colab.**

## 1. Clone repo

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD

## 2. Confirm GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 3. Install deps

In [ ]:
import torch
tv = torch.__version__.split('+')[0]
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich

## 4. Prefetch PhysioNet imagery (~2 min)

In [ ]:
!cd /content/bci-identity-leakage && python -m data.prefetch_direct --runs imagery --workers 8

## 5. Run A5 — 20 shadow EEGNets + 1 target + attack MLP

Expected wall: ~25–30 min.

In [ ]:
!cd /content/bci-identity-leakage && PYTHONUNBUFFERED=1 python -u -m experiments.08_a5_membership_inference --all

## 6. Emit run metadata + result JSON

In [ ]:
# Robust paste-back: survives a Colab reconnect and prints the result to copy.
import json, os, subprocess, datetime, platform
import torch
REPO = '/content/bci-identity-leakage'
if not os.path.isdir(REPO):
    raise SystemExit('Runtime was reset and the clone is gone. '
                     'Use Runtime > Run all to redo this notebook end to end.')
os.chdir(REPO)
EXPERIMENT, ARGS, TAG = 'experiments.08_a5_membership_inference', ['--all'], 'a5_membership_inference'
RESULTS, FIGURE = ['results/08_a5_membership_inference.json'], []
try:
    git_sha = subprocess.check_output(['git', '-C', REPO, 'rev-parse', 'HEAD']).decode().strip()
except Exception as exc:
    print(f'  git unavailable ({exc}); using "unknown"'); git_sha = 'unknown'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
now_utc = datetime.datetime.now(datetime.timezone.utc).replace(microsecond=0).isoformat().replace('+00:00', 'Z')
run_id = now_utc.replace(':', '').replace('-', '').rstrip('Z') + f'_{TAG}_{git_sha[:7]}'
meta = {'run_id': run_id, 'experiment': EXPERIMENT, 'args': ARGS, 'git_sha': git_sha,
        'hardware': f'Colab {gpu}', 'platform': platform.platform(),
        'torch_version': torch.__version__, 'completed_at_utc': now_utc,
        'outputs': RESULTS + FIGURE}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('=== run metadata ==='); print(json.dumps(meta, indent=2)); print()
missing = [r for r in RESULTS if not os.path.exists(r)]
if missing:
    raise SystemExit(f'!!! missing {missing} — the experiment cell did not finish; re-run it then this cell.')
for r in RESULTS:
    print(f'=== {r} ==='); print(open(r).read()); print()


## 7. Download artifacts

In [ ]:
# (paste-back prints the result above; no download needed)
